In [1]:
import pandas as pd
import requests
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["figure.figsize"] = (10, 7)
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm
from statsmodels.tsa.seasonal import STL
import requests
import json
import numpy as np
import re
import plotly.express as ex
from airrship.create_repertoire import (
    generate_sequence,
    load_data,
    get_genotype,
    create_allele_dict,
)
import importlib
import plotly.io as pio

tqdm.pandas()
# from Bio.Align.Applications import ClustalOmegaCommandline
# from Bio import AlignIO, SeqIO
# from Bio.SeqRecord import SeqRecord
# from Bio.Seq import Seq
import os
from scipy.stats import entropy
import matplotlib as mpl
from collections import defaultdict
import numpy as np
from VDeepJModelExperimental import (
    VDeepJAllignExperimentalSingleBeamConvSegmentationResidualRF,
)
from Trainer import SingleBeamSegmentationTrainer
import tensorflow as tf

physical_devices = tf.config.experimental.list_physical_devices("GPU")
config = tf.config.experimental.set_memory_growth(physical_devices[0], True)

<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject
2023-11-19 10:32:30.705113: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1
2023-11-19 10:33:02.142126: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2023-11-19 10:33:02.144228: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2023-11-19 10:33:02.201669: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:65:00.0 name: NVIDIA TITAN RTX computeCapability: 7.5
coreClock: 1.77GHz coreCount: 72 deviceMemorySize: 23.64GiB deviceMemoryBandwidth: 625.94GiB/s
2023-11-19 10:33:02.201709: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully o

In [2]:
mpl.rcParams["figure.figsize"] = (20, 11)
sns.set_context("poster")
tokenizer_dictionary = {
    "A": 1,
    "T": 2,
    "G": 3,
    "C": 4,
    "N": 5,
    "P": 0,  # pad token
}
import pickle


def _process_and_dpad(sequence, train=True):
    start, end = None, None
    trans_seq = [tokenizer_dictionary[i] for i in sequence]

    gap = max_length - len(trans_seq)
    iseven = gap % 2 == 0
    whole_half_gap = gap // 2

    if iseven:
        trans_seq = [0] * whole_half_gap + trans_seq + ([0] * whole_half_gap)
        if train:
            start, end = whole_half_gap, max_length - whole_half_gap - 1

    else:
        trans_seq = [0] * (whole_half_gap + 1) + trans_seq + ([0] * whole_half_gap)
        if train:
            start, end = (whole_half_gap + 1, max_length - whole_half_gap - 1)

    return trans_seq, start, end if iseven else (end + 1)


def process_sequences(self, data: pd.DataFrame, corrupt_beginning=False, verbose=False):
    padded_sequences = []
    v_start, v_end, d_start, d_end, j_start, j_end = [], [], [], [], [], []
    iterator = (
        tqdm(data.itertuples(), total=len(data)) if verbose else data.itertuples()
    )
    for row in iterator:
        seq = row.sequence
        padded_array, start, end = _process_and_dpad(seq, self.max_length)
        padded_sequences.append(padded_array)
        _adjust = start

        v_start.append(start)
        j_end.append(end)
        v_end.append(row.v_sequence_end + _adjust)
        d_start.append(row.d_sequence_start + _adjust)
        d_end.append(row.d_sequence_end + _adjust)
        j_start.append(row.j_sequence_start + _adjust)

    v_start = np.array(v_start)
    v_end = np.array(v_end)
    d_start = np.array(d_start)
    d_end = np.array(d_end)
    j_start = np.array(j_start)
    j_end = np.array(j_end)

    padded_sequences = np.vstack(padded_sequences)

    return v_start, v_end, d_start, d_end, j_start, j_end, padded_sequences


def global_genotype():
    try:
        path_to_data = importlib.resources.files("airrship").joinpath("data")
    except AttributeError:
        with importlib.resources.path("airrship", "data") as p:
            path_to_data = p
    v_alleles = create_allele_dict(f"{path_to_data}/imgt_human_IGHV.fasta")
    d_alleles = create_allele_dict(f"{path_to_data}/imgt_human_IGHD.fasta")
    j_alleles = create_allele_dict(f"{path_to_data}/imgt_human_IGHJ.fasta")

    vdj_allele_dicts = {"V": v_alleles, "D": d_alleles, "J": j_alleles}

    chromosome1, chromosome2 = defaultdict(list), defaultdict(list)
    for segment in ["V", "D", "J"]:
        allele_dict = vdj_allele_dicts[segment]
        for gene in allele_dict.values():
            for allele in gene:
                chromosome1[segment].append(allele)
                chromosome2[segment].append(allele)

    locus = [chromosome1, chromosome2]
    return locus


def decompose_call(call):
    family, G = call.split("-", 1)
    gene, allele = G.split("*")
    return family, gene, allele


locus = global_genotype()

from VDeepJUnbondedDataset import global_genotype

locus = global_genotype()
v_dict = {i.name: i.ungapped_seq.upper() for i in locus[0]["V"]}
d_dict = {i.name: i.ungapped_seq.upper() for i in locus[0]["D"]}
j_dict = {i.name: i.ungapped_seq.upper() for i in locus[0]["J"]}

v_alleles = sorted(list(v_dict))
d_alleles = sorted(list(d_dict))
j_alleles = sorted(list(j_dict))

v_allele_count = len(v_alleles)
d_allele_count = len(d_alleles)
j_allele_count = len(j_alleles)


v_allele_call_ohe = {f: i for i, f in enumerate(v_alleles)}
d_allele_call_ohe = {f: i for i, f in enumerate(d_alleles)}
j_allele_call_ohe = {f: i for i, f in enumerate(j_alleles)}

v_allele_call_rev_ohe = {i: f for i, f in enumerate(v_alleles)}
d_allele_call_rev_ohe = {i: f for i, f in enumerate(d_alleles)}
j_allele_call_rev_ohe = {i: f for i, f in enumerate(j_alleles)}


def encode_igb_v_call(v_call):
    v = np.zeros(len(v_allele_call_rev_ohe))
    for i in v_call.split(","):
        v[v_allele_call_ohe[i]] = 1
    return v

In [3]:
# for data generation
from airrship.create_repertoire import (
    generate_sequence,
    load_data,
    get_genotype,
    create_allele_dict,
)

data_dict = load_data()


def predict_sample(sample):
    eval_dataset_ = trainer.train_dataset.tokenize_sequences([sample])
    padded_seqs_tensor = tf.convert_to_tensor(eval_dataset_, dtype=tf.int32)
    dataset_from_tensors = tf.data.Dataset.from_tensor_slices(
        {
            "tokenized_sequence": padded_seqs_tensor,
            "tokenized_sequence_for_masking": padded_seqs_tensor,
        }
    )
    dataset = dataset_from_tensors.batch(1).prefetch(tf.data.AUTOTUNE)

    predicted = trainer.model.predict(dataset, verbose=True)
    return predicted


def get_v_latent_projection(sequence):
    eval_dataset_ = trainer.train_dataset.tokenize_sequences([sequence])
    v_mask_input_embedding = trainer.model.concatenated_v_mask_input_embedding(
        eval_dataset_
    )
    v_feature_map = trainer.model._encode_masked_v_signal(v_mask_input_embedding)

    v_allele_latent = trainer.model.v_allele_mid(v_feature_map)
    v_allele_latent = dec.transform(v_allele_latent.numpy())
    return v_allele_latent[0]


def getting_padding_size(seq, max_length=512):
    start, end = None, None
    gap = max_length - len(seq)
    iseven = gap % 2 == 0
    whole_half_gap = gap // 2

    if iseven:
        start, end = whole_half_gap, whole_half_gap

    else:
        start, end = whole_half_gap + 1, whole_half_gap
    return start, end


def log_threshold(prediction, th=0.4):
    ast = np.argsort(prediction)[::-1]
    R = [ast[0]]
    for ip in range(1, len(ast)):
        DIFF = np.log(prediction[ast[ip - 1]] / prediction[ast[ip]])
        if DIFF < th:
            R.append(ast[ip])
        else:
            break
    return R


def extract_prediction_alleles(probabilites, th=0.4):
    V_ratio = []
    for v_all in probabilites:
        v_alleles = log_threshold(v_all, th=th)
        V_ratio.append([v_allele_call_rev_ohe[i] for i in v_alleles])
    return V_ratio

In [4]:
def dynamic_cumulative_confidence_threshold(prediction, percentage=0.9):
    sorted_indices = np.argsort(prediction)[::-1]
    selected_labels = []
    cumulative_confidence = 0.0

    total_confidence = sum(prediction)
    threshold = percentage * total_confidence

    for idx in sorted_indices:
        cumulative_confidence += prediction[idx]
        selected_labels.append(idx)

        if cumulative_confidence >= threshold:
            break

    return selected_labels


def extract_prediction_alleles_dynamic_sum(probabilites, percentage=0.9):
    V_ratio = []
    for v_all in tqdm(probabilites):
        v_alleles = dynamic_cumulative_confidence_threshold(
            v_all, percentage=percentage
        )
        V_ratio.append([v_allele_call_rev_ohe[i] for i in v_alleles])
    return V_ratio

In [ ]:
from Models import AlignAIRR
from Trainer import Trainer

trainer = Trainer(
    model= AlignAIRR,
    data_path = "/localdata/alignairr_data/AlignAIRR_Large_Train_Dataset/AlignAIRR_Large_Train_Dataset_SeqSimulator.csv",
    batch_read_file=True,
    epochs=1,
    batch_size=32,
    steps_per_epoch=150_000,
    verbose=1,
    allele_map_path='',

)
trainer.model.build({'tokenized_sequence':(512,1)})
trainer.model.load_weights("/localdata/alignairr_data/AlignAIRR_Refactored/evaluation_checkpoints/model_acc_0.9953/variables/variables")
model = trainer.model